# Lilly v2 — reader pass-12

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian only (č ć đ š ž). No Cyrillic.

Pass-7c attached Commons harvest, EasyOCR-cropped it, and drowned the human
labels. Photographs got worse. **Do not relaunch pass-7c.**

Pass-8 mixed 1,294 human crops ×2 with **18,045** sign-letter plates (~12%
human). Crop gate refused: real-crop words 41.7% → 44.7%, Bosnian letters
64.0% → 60.0%. **Do not relaunch pass-8.**

Pass-9 capped plates at 50% human but **kept the plates with the most čćđšž**,
so the remaining half was denser letter-font gradient than pass-8. Same gate:
words 41.7% → 45.5%, letters **64.0% → 60.0% again** (16/25 → 15/25). **Do
not relaunch pass-9.**

Pass-10 trained human-only (1,294 ×4, plates in valid only). Crop gate
refused **worse than shipped**: real words 41.7% → **39.4%**, letters 16/25 →
**14/25**. The two new misses were `iznad česme` (č → garbage) and one of
four `PUTNIČKI` (Č → C). Human-only overfit the same 1,294 crops. **Do not
relaunch pass-10 as a single mix.**

Pass-11 ran 31 Aug 2026 (kernel ERROR, git `6310d90`, T4, 5m39s). Two
trainings, not mixed: (1) 18,045 plates, 2 epochs, checkpoint; (2) 1,294
human, 3 epochs from that checkpoint. Stage 1: words 41.7% → 42.4%,
letters 64.0% stayed. Stage 2: words 42.4% → **41.7%**, letters **64.0% →
60.0%** (16/25 → 15/25). One of four `PUTNIČKI` Č → C — same spend as
pass-10. Crop gate refused. No photograph score. No `lilly-read.zip`.
**Do not relaunch pass-11.** `training/RESULTS-ocr-pass11.md`.

Attach (all required): `lilly-read-pass1`, `lilly-ocr-crops`,
`lilly-ocr-sign-letters`.

Photograph gate before `lilly-read.zip`: beat shipped 54.7 / 45.0 / 44%
diacritic / ≤180 invented. Setup mistakes and gate failures stop the kernel.

Offload contract (`training/kaggle_offload.py`): Output always holds
`stdout.txt`, `experiment_log.json`, `metrics.jsonl`. COMPLETE + zip is still
not install. Not a Kaggle competition — do not submit.

In [ ]:
# 1. Stop here unless the machine is actually set up
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)
print("network ok")

# Under /kaggle/working, so it is Output and outlives the log.
# agentic-kaggle-skill offload: the tee is stdout.txt, not a child fd Kaggle never sees.
TEE = Path("/kaggle/working/stdout.txt")

def run(*cmd, quiet=False):
    """Run a child process and put its output somewhere it can be found.

    Kaggle's log holds what this notebook process prints. A child process
    writing to its own stdout is not in it. Pass-7b and pass-7c both logged
    `$ python3 ... train_ocr.py` and then nothing whatsoever until the zip 21
    minutes later, and `python -u` on the child did not fix it, because the
    child's file descriptor never reaches the log to begin with. Both runs
    therefore ended with their before/after rates and the gate's own verdict
    unrecoverable, and pass-7c had to be re-measured on a laptop to discover it
    read photographs worse. Read the child's output here and reprint it, and
    tee it into Output so the numbers survive a dropped log too.
    """
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)

In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working
# Everything under /kaggle/working becomes Output. Training copies tens of
# thousands of PNGs into data/ocr/train and valid, and with the clone there too,
# `kaggle kernels output` spent 172 s on PNGs and git objects and never reached
# the weights zip at all. Only the zips below belong in Output.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "train_ocr.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zips")
from training.kaggle_offload import Offload
OFF = Offload("ocr", os.environ.get("LILLY_RUN_ID", "ocr"))
OFF.hardware(torch.cuda.get_device_name(0))

In [ ]:
# 3. Install what we need (~2 min)
# Do NOT pip-install torch from requirements.txt — that file pins CPU wheels for
# the Mac. Kaggle already ships a GPU build; replacing it wastes time and can
# break CUDA.
NEEDED = ["easyocr", "opencv-python-headless", "pillow", "huggingface_hub"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line.startswith("--"):
        continue
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# 3b. Prove the GPU can backprop before an hour is spent generating images
x = torch.randn(256, 256, device="cuda", requires_grad=True)
y = (x @ torch.randn(256, 256, device="cuda")).sum()
y.backward()
print(f"GPU backprop ok on {torch.cuda.get_device_name(0)}")
torch.cuda.empty_cache()

In [ ]:
# 4. Base reader weights from Hugging Face + pass-1 from the attached dataset
# Dataset may land as files, as read/lilly.pth, or as read.zip (`-r zip`).
import shutil, zipfile
run("python3", "scripts/fetch_models.py")
READ = Path("models/lilly/read")
base = READ / "latin_g2.pth"
assert base.is_file() and base.stat().st_size > 1_000_000, "fetch_models missing read weights"
net = READ / "user_network"
for needed in ("lilly.yaml", "lilly.py"):
    assert (net / needed).is_file(), f"fetch_models missing user_network/{needed}"

input_root = Path("/kaggle/input")
print("attached inputs:",
      [str(p.relative_to(input_root)) for p in input_root.rglob("*")][:40]
      if input_root.is_dir() else "NONE")

unpack = Path("/tmp/pass1-unpack")
if unpack.exists():
    shutil.rmtree(unpack)
unpack.mkdir()
if input_root.is_dir():
    for z in input_root.rglob("*.zip"):
        dest = unpack / z.stem
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print("unpacked", z)

hits = []
for root in (input_root, unpack):
    if root.is_dir():
        hits.extend(root.rglob("lilly.pth"))
if not hits:
    raise SystemExit(
        "No pass-1 weights under /kaggle/input. Attach lilly-read-pass1 "
        "or relaunch: python3 scripts/kaggle_train.py ocr")
src = hits[0]
shutil.copy(src, READ / "lilly.pth")
un = src.parent / "user_network"
if un.is_dir():
    shutil.copytree(un, net, dirs_exist_ok=True)
for name in ("lilly.yaml", "lilly.py"):
    sidecar = src.parent / name
    if sidecar.is_file():
        shutil.copy(sidecar, net / name)
INIT = READ / "lilly.pth"
print(f"starting reader from {src} -> {INIT}")
assert INIT.is_file() and INIT.stat().st_size > 100_000
print("pass-11 will continue from the shipped reader at", INIT)

In [ ]:
# 5. Real hand-labelled crops FIRST — prepare_ocr_data deletes syn* files.
# If synthetic is written first, merging real wipes the generator images
# and train collapses to the 1,294 Latin crops (Kaggle pass-3 v5, ~2 min).
# Real names are not syn*; sign-letter plates then keep them.
crops_dir = Path("data/ocr/crops")
crops_dir.mkdir(parents=True, exist_ok=True)
copied = 0
for root in (input_root, unpack):
    if not root.is_dir():
        continue
    for png in root.rglob("*.png"):
        path = png.as_posix().lower()
        # Harvested full photos and sign-letter syn* plates are other datasets.
        if "harvest" in path or "sign-letters" in path:
            continue
        if png.name.startswith("syn"):
            continue
        # crops2/ PNGs are handled separately below; skip them here.
        if "/crops2/" in png.as_posix():
            continue
        dest = crops_dir / png.name
        if not dest.exists():
            shutil.copy(png, dest)
            copied += 1
print(f"copied {copied} real crop PNGs from lilly-ocr-crops -> {crops_dir}")

# Pass-12: crops2/ PNGs ship under the crops2/ prefix in the zip.
# populate data/ocr/crops2/ so prepare_ocr_data.py can find them via crops2/labels-human.tsv.
crops2_dir = Path("data/ocr/crops2")
crops2_dir.mkdir(parents=True, exist_ok=True)
copied2 = 0
for root in (input_root, unpack):
    if not root.is_dir():
        continue
    c2 = root / "crops2"
    if not c2.is_dir():
        continue
    for png in c2.glob("*.png"):
        dest = crops2_dir / png.name
        if not dest.exists():
            shutil.copy(png, dest)
            copied2 += 1
print(f"copied {copied2} crops2 PNGs from lilly-ocr-crops/crops2 -> {crops2_dir}")

merged = False
for labels in (Path("data/ocr/crops/labels-human.tsv"),
               Path("data/ocr/crops2/labels-human.tsv")):
    if not labels.is_file():
        continue
    rows = [l for l in labels.read_text(encoding="utf-8").splitlines() if l.strip()]
    if not rows:
        continue
    img_name = rows[0].split("\t")[0]
    img_path = labels.parent / img_name
    if img_path.is_file():
        run("python3", "training/prepare_ocr_data.py", "--labels", str(labels))
        print(f"merged real crops from {labels}")
        merged = True
if not merged:
    raise SystemExit(
        "Pass-12 needs real crop PNGs. Attach lilly-ocr-crops "
        "or relaunch: python3 scripts/kaggle_train.py ocr")
print("real Latin crops are in train/valid; sign-letter plates next (keeps these)")


In [ ]:
# 5b. Sign-letter plates on top of the real crops.
# prepare_ocr_data keeps files whose names are not syn* (the real crops),
# then copies these syn* plates in. Do not generate 50k on this box — the
# plates were drawn locally from the Bosnian corpus (especially đ) and
# attached as lilly-ocr-sign-letters.
sign_dir = Path("data/ocr/sign-letters")
sign_dir.mkdir(parents=True, exist_ok=True)
labels_hit = None
for root in (input_root, unpack):
    if not root.is_dir():
        continue
    for labels in root.rglob("labels.tsv"):
        path = labels.as_posix().lower()
        if "harvest" in path:
            continue
        head = labels.read_text(encoding="utf-8")[:200]
        if "syn0" not in head:
            continue
        labels_hit = labels
        break
    if labels_hit is not None:
        break
if labels_hit is None:
    raise SystemExit(
        "Sign-letter plates are required for this pass. "
        "Attach lilly-ocr-sign-letters or relaunch: python3 scripts/kaggle_train.py ocr")

src_dir = labels_hit.parent
copied_syn = 0
for line in labels_hit.read_text(encoding="utf-8").splitlines():
    parts = line.split("\t")
    if len(parts) < 2:
        continue
    src = src_dir / parts[0]
    if not src.is_file():
        continue
    dest = sign_dir / parts[0]
    if not dest.exists():
        shutil.copy(src, dest)
        copied_syn += 1
shutil.copy(labels_hit, sign_dir / "labels.tsv")
print(f"copied {copied_syn} sign-letter PNGs from lilly-ocr-sign-letters -> {sign_dir}")
if copied_syn < 5000:
    raise SystemExit(f"sign-letters too thin: {copied_syn} pngs (need ≥5000)")

run("python3", "training/prepare_ocr_data.py", "--labels", str(sign_dir / "labels.tsv"))
train_gt = Path("data/ocr/train/gt.txt")
valid_gt = Path("data/ocr/valid/gt.txt")
n_syn = sum(1 for row in train_gt.read_text(encoding="utf-8").splitlines()
            if row.startswith("syn"))
print(f"sign-letter syn* train rows: {n_syn}")
assert n_syn > 4000, f"prepare did not keep sign letters: {n_syn}"

In [ ]:
# 5c. Pass-7c generated photo-style synthetic here. Do not. That mix plus
# harvest auto-crops passed the crop gate and lost on photographs.
n_photo = len(list(Path("data/ocr/train").glob("photo*.png")))
print(f"photo-style train files (must stay 0): {n_photo}")
assert n_photo == 0, f"photo-style synthetic leaked into pass-8: {n_photo}"

In [ ]:
# 5d. Pass-7c EasyOCR-cropped Commons harvest into train. Do not.
# Those auto-crops drowned 1,294 human labels and invented more words on
# brick/foliage. If the old dataset is still attached, ignore it.
harvest_attached = False
for root in (input_root, unpack):
    if root.is_dir() and any("ocr-harvest" in p.as_posix() for p in root.rglob("*")):
        harvest_attached = True
        break
if harvest_attached:
    print("lilly-ocr-harvest is attached and will be ignored "
          "(pass-8 is not a harvest pass)", flush=True)
auto_n = len(list(Path("data/ocr/train").glob("auto_*")))
assert auto_n == 0, f"harvest auto-crops leaked into train: {auto_n}"
print("no harvest auto-crops in train")

In [ ]:
# 5e. Split train into two lists. Do not mix them. Stage 1 writes plates
# only; stage 2 writes human only. Valid keeps both (the syn floor).
lines = [l for l in train_gt.read_text(encoding="utf-8").splitlines() if l.strip()]
def is_human(row):
    name = row.split("\t", 1)[0]
    return not name.startswith("syn") and not name.startswith("photo") and not name.startswith("auto_")
human_lines = [l for l in lines if is_human(l)]
plate_lines = [l for l in lines if l.split("\t", 1)[0].startswith("syn")]
assert human_lines, "no human-labelled crops in train — the merge did not stick"
assert len(plate_lines) > 4000, f"plates too thin: {len(plate_lines)}"
assert not any(l.split("\t", 1)[0].startswith("syn") for l in human_lines)
assert all(l.split("\t", 1)[0].startswith("syn") for l in plate_lines)
valid_n = sum(1 for _ in open(valid_gt, encoding="utf-8"))
print(f"two stages, not mixed: {len(plate_lines)} plates then {len(human_lines)} human  |  valid {valid_n:,}")
assert valid_n > 100, f"only {valid_n} valid crops"
OFF.metric("plate_n", len(plate_lines), stage="split")
OFF.metric("human_n", len(human_lines), stage="split")


In [ ]:
# 5f. Four real GPU training steps before the long run (~30 s)
# Caught the cuda/cpu CTCLoss bug that killed v1 at step 1 after 2 min of setup.
run("python3", "training/train_ocr.py", "--quick-test")
print("GPU OCR training smoke ok")

In [ ]:
# 6. TWO TRAININGS, NOT MIXED — pass-13.
# Stage 1: plates only, checkpoint, no install (crop gate would refuse plates).
# Stage 2: human only, resume stage 1, crop gate, then cell 6b photographs.
# Pass-10 5-epoch human-only overfit. Stage 2 is 3 epochs, each crop once.
os.environ["LILLY_RUN_ID"] = "heavy-pass13"
os.environ["PYTHONUNBUFFERED"] = "1"
OFF.body["run_id"] = "heavy-pass13"
OFF.flush()
STAGE1 = Path("models/lilly/read-stage1.pth")
TRAINED = Path("models/lilly/read-trained.pth")

train_gt.write_text("\n".join(plate_lines) + "\n", encoding="utf-8")
assert all(l.split("\t", 1)[0].startswith("syn") for l in plate_lines)
print(f"stage 1 train: {len(plate_lines)} plates, 0 human")
try:
    run("python3", "-u", "training/train_ocr.py",
        "--epochs", "2", "--batch-size", "16",
        "--lr", "3e-6", "--grad-clip", "1.0", "--warmup-frac", "0.05",
        "--weights", str(INIT),
        "--no-install", "--checkpoint", str(STAGE1))
except subprocess.CalledProcessError:
    OFF.fail("stage 1 train_ocr exit 1 — collapse")
    raise
assert STAGE1.is_file() and STAGE1.stat().st_size > 100_000, "stage 1 wrote no weights"
OFF.check_trainproof()

train_gt.write_text("\n".join(human_lines) + "\n", encoding="utf-8")
assert all(not l.split("\t", 1)[0].startswith("syn") for l in human_lines)
print(f"stage 2 train: {len(human_lines)} human, 0 plates")
try:
    run("python3", "-u", "training/train_ocr.py",
        "--epochs", "1", "--batch-size", "16",
        "--lr", "1e-6", "--grad-clip", "1.0", "--warmup-frac", "0.05",
        "--weights", str(STAGE1), "--keep-trained", str(TRAINED))
except subprocess.CalledProcessError:
    OFF.fail("train_ocr exit 1 — crop gate or collapse")
    raise
assert TRAINED.is_file() and TRAINED.stat().st_size > 100_000, (
    "training wrote no weights")
OFF.check_trainproof()
run("zip", "-j", "/kaggle/working/lilly-read-trained.zip", str(TRAINED))
print("lilly-read-trained.zip saved")


In [ ]:
# 6b. The gate that measures what the README claims
# train_ocr.py decides on crops. Pass-7c passed that crop gate while reading
# photographs worse. The photographs decide here, before lilly-read.zip.
import json

SHIPPED = {"per_photo": 54.7, "pooled": 45.0, "diacritic": 44.0, "invented": 180}
METRICS = Path("/kaggle/working/ocr-photo-gate.json")

# The 40 renderings are not in git — 13 MB of other people's photographs. The
# answer key and scored-sources.tsv are, so they can be rebuilt from the URLs.
run("python3", "data/scripts/restore_scored_photos.py")
run("python3", "-u", "training/evaluate_ocr.py",
    "--out", "training/RESULTS-ocr-pass13.md", "--json", str(METRICS),
    "--cache", "/kaggle/temp/reader-output-pass13.json")

got = json.loads(METRICS.read_text())
print(f"\nper photograph {got['per_photo']:.1f}% (shipped {SHIPPED['per_photo']}%)"
      f"  |  pooled {got['pooled']:.1f}% (shipped {SHIPPED['pooled']}%)"
      f"  |  diacritic {got['diacritic']:.1f}% (shipped {SHIPPED['diacritic']}%)"
      f"  |  invented {got['invented']} (shipped {SHIPPED['invented']})")
for key in ("per_photo", "pooled", "diacritic", "invented"):
    OFF.metric(key, got[key], stage="photograph-gate")
worse = (
    got["per_photo"] < SHIPPED["per_photo"]
    or got["pooled"] < SHIPPED["pooled"]
    or got["diacritic"] < SHIPPED["diacritic"]
    or got["invented"] > SHIPPED["invented"]
)
if worse:
    OFF.fail("photograph gate: worse than shipped")
    raise SystemExit(
        "this reader reads photographs worse than the one already shipped — "
        "not packaging it. The weights are in lilly-read-trained.zip for "
        "inspection; lilly-read.zip is deliberately absent.")
print("photograph gate passed")

In [ ]:
# 7. Package what the app loads — only when the install gate passed
READ = Path("models/lilly/read")
NET = READ / "user_network"
for needed in ("lilly.yaml", "lilly.py"):
    assert (NET / needed).is_file(), f"missing user_network/{needed}"

assert (READ / "lilly.pth").is_file(), "gate passed but lilly.pth missing"
run("zip", "-qr", "/kaggle/working/lilly-read.zip",
    "models/lilly/read/lilly.pth",
    "models/lilly/read/user_network/lilly.yaml",
    "models/lilly/read/user_network/lilly.py")
size = Path("/kaggle/working/lilly-read.zip").stat().st_size
assert size > 100_000, f"zip too small: {size}"
print(f"lilly-read.zip — {size / 1048576:.1f} MB")
OFF.finish("passed", [
    "/kaggle/working/lilly-read.zip",
    "/kaggle/working/lilly-read-trained.zip",
    "/kaggle/working/stdout.txt",
    "/kaggle/working/experiment_log.json",
])

out = sorted(Path("/kaggle/working").rglob("*"))
print(f"\nOutput holds {len(out)} entries:")
for p in out[:20]:
    print(f"  {p.relative_to('/kaggle/working')}  {p.stat().st_size / 1048576:.1f} MB")
assert len(out) < 50, f"Output has {len(out)} entries — the fetch will drown"


**If the kernel is COMPLETE and `lilly-read.zip` is in Output:** unzip it over the repo root.
Keep `latin_g2.pth` and any previous `lilly.pth` as backup.

**If the kernel is ERROR, or COMPLETE without `lilly-read.zip`:** do not install
`lilly-read-trained.zip`. The photograph gate refused, or training failed.
Fix that failure, then relaunch. Do not relaunch pass-7c, pass-8, pass-9,
pass-10, or pass-11.